# 02 — Path B Step 1: Download NSDUH Public Use Files

Download annual NSDUH PUF files (Stata format) for 2002–2023.
Each file is ~15–30 MB compressed, ~55,000–70,000 respondents.

**Output:** Stata files in `data/raw/nsduh/pufs/{year}/`

In [1]:
import os, requests, zipfile, io, time, re
from pathlib import Path
from bs4 import BeautifulSoup

DATA_DIR = Path(os.path.abspath(os.path.join('..', 'data')))
PUF_DIR = DATA_DIR / 'raw' / 'nsduh' / 'pufs'
PUF_DIR.mkdir(parents=True, exist_ok=True)
print(f'PUF directory: {PUF_DIR}')

PUF directory: /home/abie/ai_assisted_us_health_data_analysis/data/raw/nsduh/pufs


## Discover Download URLs

Scrape each year's SAMHSA data page to find the Stata PUF download link.
Stata format is the most compact (~15-30 MB per year).

In [2]:
YEARS = list(range(2002, 2024))  # 2002–2023

def find_puf_links(year):
    """Scrape SAMHSA PUF page for download links."""
    url = f'https://www.samhsa.gov/data/data-we-collect/nsduh-national-survey-drug-use-and-health/datafiles/{year}'
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, 'lxml')
    links = {}
    for a in soup.find_all('a', href=True):
        href = a['href']
        text = a.get_text(strip=True).lower()
        if not href.endswith('.zip'):
            continue
        full = href if href.startswith('http') else f'https://www.samhsa.gov{href}'
        if 'stata' in text and 'setup' not in text:
            links['stata'] = full
        elif ('delimited' in text or 'tsv' in text) and 'setup' not in text:
            links['tsv'] = full
        elif 'sas' in text and 'setup' not in text:
            links['sas'] = full
    # Also check href for format hints
    if 'stata' not in links:
        for a in soup.find_all('a', href=True):
            href = a['href']
            if href.endswith('.zip') and 'stata' in href.lower():
                full = href if href.startswith('http') else f'https://www.samhsa.gov{href}'
                links['stata'] = full
    return links

# Discover all links
all_links = {}
for year in YEARS:
    print(f'{year}: ', end='')
    try:
        links = find_puf_links(year)
        all_links[year] = links
        fmt = 'stata' if 'stata' in links else (list(links.keys())[0] if links else 'NONE')
        print(f'{fmt} ✓' if links else 'no links found')
    except Exception as e:
        print(f'error: {e}')
    time.sleep(0.3)

print(f'\nFound links for {len(all_links)} years')

2002: 

stata ✓


2003: 

stata ✓


2004: 

...

2022: 

stata ✓


2023: 

stata ✓



Found links for 22 years


In [3]:
# Build download plan — prefer Stata, fall back to TSV or SAS
download_plan = {}
for year, links in sorted(all_links.items()):
    for fmt in ['stata', 'tsv', 'sas']:
        if fmt in links:
            download_plan[year] = {'format': fmt, 'url': links[fmt]}
            break

print(f'Download plan: {len(download_plan)} years')
for year, info in sorted(download_plan.items()):
    print(f"  {year}: {info['format']}")

Download plan: 22 years
  2002: stata
  2003: stata
  2004: stata
  2005: stata
  2006: stata
  2007: stata
  2008: stata
  2009: stata
  2010: stata
  2011: stata
  2012: stata
  2013: stata
  2014: stata
  2015: stata
  2016: stata
  2017: stata
  2018: stata
  2019: stata
  2020: stata
  2021: stata
  2022: stata
  2023: stata


## Download and Extract

In [4]:
def download_puf(year, url, dest_dir):
    """Download and extract a PUF ZIP file."""
    year_dir = dest_dir / str(year)
    # Check if already downloaded (case-insensitive extensions for Linux)
    if year_dir.exists():
        data_files = list(year_dir.rglob('*.dta')) + list(year_dir.rglob('*.DTA')) + \
                     list(year_dir.rglob('*.tsv')) + list(year_dir.rglob('*.sas7bdat'))
        if data_files:
            return 'exists'
    
    try:
        r = requests.get(url, timeout=600)
        r.raise_for_status()
        year_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
            zf.extractall(year_dir)
        size_mb = len(r.content) / 1024 / 1024
        return f'ok ({size_mb:.1f} MB)'
    except Exception as e:
        return f'FAILED: {e}'

# Download all
for year, info in sorted(download_plan.items()):
    print(f"{year}: ", end='')
    result = download_puf(year, info['url'], PUF_DIR)
    print(result)
    if result.startswith('ok'):
        time.sleep(1)  # Be polite

2002: exists
2003: exists
2004: exists
2005: exists
2006: exists
2007: exists
2008: exists
2009: exists
2010: exists
2011: exists
2012: exists
2013: exists
2014: exists
2015: exists
2016: exists
2017: exists
2018: exists
2019: exists
2020: exists
2021: exists
2022: exists
2023: exists


In [5]:
# Verify downloads (case-insensitive extensions for Linux)
print('Downloaded PUF files:\n')
for year_dir in sorted(PUF_DIR.iterdir()):
    if not year_dir.is_dir() or not year_dir.name.isdigit():
        continue
    data_files = list(year_dir.rglob('*.dta')) + list(year_dir.rglob('*.DTA')) + \
                 list(year_dir.rglob('*.tsv')) + list(year_dir.rglob('*.sas7bdat')) + \
                 list(year_dir.rglob('*.txt'))
    big = [f for f in data_files if f.stat().st_size > 1_000_000]
    if big:
        largest = max(big, key=lambda f: f.stat().st_size)
        print(f'  {year_dir.name}: {largest.name} '
              f'({largest.stat().st_size/1024/1024:.0f} MB)')
    else:
        print(f'  {year_dir.name}: no data files found')

total_years = sum(1 for d in PUF_DIR.iterdir() 
                  if d.is_dir() and d.name.isdigit())
print(f'\nTotal: {total_years} years downloaded')

Downloaded PUF files:

  2002: NSDUH_2002.DTA (140 MB)
  2003: NSDUH_2003.DTA (150 MB)
  2004: NSDUH_2004.DTA (168 MB)
  2005: NSDUH_2005.DTA (179 MB)
  2006: NSDUH_2006.DTA (189 MB)
  2007: NSDUH_2007.DTA (189 MB)
  2008: NSDUH_2008.DTA (190 MB)
  2009: NSDUH_2009.DTA (188 MB)
  2010: NSDUH_2010.DTA (196 MB)
  2011: NSDUH_2011.DTA (200 MB)
  2012: NSDUH_2012.DTA (189 MB)
  2013: NSDUH_2013.DTA (189 MB)
  2014: NSDUH_2014.DTA (190 MB)
  2015: NSDUH_2015.DTA (173 MB)
  2016: NSDUH_2016.DTA (171 MB)
  2017: NSDUH_2017.DTA (168 MB)
  2018: NSDUH_2018.DTA (171 MB)
  2019: NSDUH_2019.DTA (173 MB)
  2020: NSDUH_2020.dta (106 MB)
  2021: NSDUH_2021.DTA (186 MB)
  2022: NSDUH_2022.dta (168 MB)
  2023: NSDUH_2023.dta (160 MB)

Total: 22 years downloaded


## Summary

PUF files are now in `data/raw/nsduh/pufs/{year}/`.

**Next:** [03_puf_analysis.ipynb](03_puf_analysis.ipynb) — Analyze the microdata